# 08 — Retrieval hybride, MMR et reranking

Suit une question à travers dense, BM25, fusion RRF, diversification MMR et cross-encoder BGE Reranker.

> Vérifiez toujours les drapeaux `RUN_*` avant l'exécution. Pour cette reconstruction propre, les traitements locaux nécessaires des étapes 02 à 07 sont déjà activés ; les collectes réseau et évaluations GPU facultatives restent désactivées.


In [1]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('Données actives :', PROJECT_ROOT / 'data')


Projet : <PROJECT_ROOT>
Python : c:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs\RAG ENSEA\.venv\Scripts\python.exe
Données actives : <PROJECT_ROOT>\data


## Chaîne de recherche

1. BGE-M3 encode la question.
2. LanceDB, BM25 et le canal des lignes tabulaires proposent leurs candidats.
3. Les meilleurs documents servent de semences à une recherche locale de la meilleure page, avec pénalité si période, indicateur ou type de mesure ne correspondent pas.
4. RRF fusionne les rangs.
5. MMR sélectionne jusqu'à 15 passages diversifiés.
6. BGE Reranker réordonne les candidats et retourne normalement 5 passages.
7. Une demande explicitement exhaustive ou formulée comme une collection (« quels sont les X ») peut en conserver jusqu'à 10.

En mode exhaustif seulement, les bassins dense/BM25 et le lot soumis au reranker sont élargis. Un rappel de couverture identifie aussi automatiquement la catégorie demandée dans des formulations générales comme « quels sont les X » ou « liste des X » : les passages contenant ce terme rejoignent le reranker, sans être forcés dans la réponse. MMR et la déduplication tolèrent davantage les documents proches, car deux passages très similaires peuvent décrire deux éléments distincts d'une même liste. Le budget du prompt est ensuite partagé entre tous les résultats retenus.


In [2]:
import pandas as pd
from config import load_config
from conversation import contextualize_question, is_exhaustive_request

config = load_config(PROJECT_ROOT / 'config.yaml')
history = [
    {'role': 'user', 'content': "Quels sont les finalistes du concours d'éloquence du CDR ?"},
    {'role': 'assistant', 'content': 'Plusieurs finalistes ont été retrouvés.'},
]
followup = 'La liste est-elle exhaustive ? Je veux la liste complète.'
retrieval_query, contextualized = contextualize_question(followup, history)
print(retrieval_query)
print('Question contextualisée :', contextualized)
print('Recherche élargie :', is_exhaustive_request(followup))


Question précédente : Quels sont les finalistes du concours d'éloquence du CDR ?
Référence résolue au tour précédent : Plusieurs finalistes ont été retrouvés.
Question de suivi : La liste est-elle exhaustive ? Je veux la liste complète.
Question contextualisée : True
Recherche élargie : True


In [3]:
from source_router import route_question

route_examples = [
    "Quelles sont les filières de l'ENSEA ?",
    "Quel est le PIB de la Côte d'Ivoire selon l'ANSTAT ?",
    "Quels sont les finalistes du concours du CDR ?",
]
display(pd.DataFrame([
    {
        'question': question,
        'route': route_question(question).route,
        'raison': route_question(question).reason,
    }
    for question in route_examples
]))


,question,route,raison
0,Quelles sont les filières de l'ENSEA ?,institutional,connaissance institutionnelle stable ou admini...
1,Quel est le PIB de la Côte d'Ivoire selon l'AN...,economic,source économique explicite : anstat
2,Quels sont les finalistes du concours du CDR ?,facebook,structure ou source étudiante explicite : cdr


In [4]:
RUN_RETRIEVAL = True
if RUN_RETRIEVAL:
    from rag_pipeline import RAGPipeline
    pipeline = RAGPipeline(config)
    max_results = config.retrieval.exhaustive_max_results if is_exhaustive_request(followup) else None
    results = pipeline.retriever.retrieve(retrieval_query, max_results=max_results)
    display(pd.DataFrame([{
        'rang': index,
        'titre': chunk.title,
        'section': chunk.section,
        'route': pipeline.retriever.last_decision.route,
        'dense': chunk.dense_score,
        'bm25': chunk.bm25_score,
        'rerank': chunk.rerank_score,
        'url': chunk.url,
    } for index, chunk in enumerate(results, 1)]))
    display(pd.Series(pipeline.retriever.last_timings).to_frame('valeur'))
else:
    print('Chargement GPU désactivé. Passez RUN_RETRIEVAL à True pour exécuter la recherche.')

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


,rang,titre,section,route,dense,bm25,rerank,url
0,1,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,facebook,0.534220,23.698209,0.959078,https://www.facebook.com/cdr.ensea.1/posts/pfb...
1,2,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,facebook,0.532898,23.499113,0.950227,https://www.facebook.com/cdr.ensea.1/posts/pfb...
2,3,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,facebook,0.544459,23.499113,0.944177,https://www.facebook.com/cdr.ensea.1/posts/pfb...
3,4,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,facebook,0.533843,29.112741,0.884239,https://www.facebook.com/cdr.ensea.1/posts/pfb...
4,5,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,facebook,0.541950,22.437350,0.772187,https://www.facebook.com/cdr.ensea.1/posts/pfb...
5,6,cdr.ensea.1 — 🎤✨ Le CEE vous présente son quat...,cdr.ensea.1 — 🎤✨ Le CEE vous présente son quat...,facebook,0.523485,15.981893,0.580233,https://www.facebook.com/cdr.ensea.1/posts/pfb...
6,7,cdr.ensea.1 — 🎤✨ Revivons ensemble une finale ...,cdr.ensea.1 — 🎤✨ Revivons ensemble une finale ...,facebook,0.554380,13.920125,0.952926,https://www.facebook.com/cdr.ensea.1/posts/pfb...


,valeur
query_embedding_seconds,0.059199
dense_search_seconds,0.038166
bm25_search_seconds,0.001327
table_fact_search_seconds,0.001033
page_rescue_seconds,0.001088
fusion_metadata_seconds,0.001175
vector_lookup_mmr_seconds,0.074432
rerank_seconds,4.094
total_seconds,4.27318
dense_candidates,31


In [5]:
if RUN_RETRIEVAL:
    debug = pd.DataFrame(pipeline.retriever.last_debug_rows)
    columns = [name for name in ['title', 'url', 'content_type', 'page', 'indicator', 'period', 'measure_type', 'dense_rank', 'bm25_rank', 'table_fact_rank', 'page_rescue_rank', 'page_rescue_bonus', 'rrf_rank', 'mmr_rank', 'rerank_rank', 'rerank_score', 'selection_score', 'anchor_candidate', 'anchor_title_candidate', 'coverage_reserved', 'final_rank'] if name in debug]
    display(debug[columns].sort_values(['final_rank', 'rerank_rank'], na_position='last').head(30))


,title,url,content_type,page,indicator,period,measure_type,dense_rank,bm25_rank,table_fact_rank,...,page_rescue_bonus,rrf_rank,mmr_rank,rerank_rank,rerank_score,selection_score,anchor_candidate,anchor_title_candidate,coverage_reserved,final_rank
3,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,text,None,,,,10,2,None,...,0.0,4,5,1,0.959078,1.109078,True,True,True,1.0
7,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,text,None,,,,12,4,None,...,0.0,8,8,2,0.950227,1.100227,True,True,True,2.0
1,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,text,None,,,,6,3,None,...,0.0,2,2,3,0.944177,1.094177,True,True,True,3.0
2,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,text,None,,,,11,1,None,...,0.0,3,3,4,0.884239,1.034239,True,True,True,4.0
4,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,text,None,,,,7,5,None,...,0.0,5,4,6,0.772187,0.922187,True,True,True,5.0
10,cdr.ensea.1 — 🎤✨ Le CEE vous présente son quat...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,text,None,,,,14,8,None,...,0.0,11,9,11,0.580233,0.730233,True,True,True,6.0
5,cdr.ensea.1 — 🎤✨ Revivons ensemble une finale ...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,text,None,,,,4,9,None,...,0.0,6,7,5,0.952926,0.952926,False,False,False,7.0
26,cdr.ensea.1 — 🎤✨ Quand la rigueur du droit ren...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,text,None,,,,24,23,None,...,0.0,27,27,7,0.824179,0.824179,False,False,False,NaN
8,cdr.ensea.1 — 🎤✨La présélection du Concours EN...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,text,None,,,,5,15,None,...,0.0,9,10,8,0.812272,0.812272,False,False,False,NaN
6,cdr.ensea.1 — 🎤✨ Une demi-finale enflammée s’e...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,text,None,,,,3,12,None,...,0.0,7,6,9,0.787114,0.787114,False,False,False,NaN


## Validation de la relation indicateur–période–valeur

Le bonus de facettes reste générique : il ne connaît aucune réponse. Il valorise un chunk
`table_fact` ou `derived_fact` lorsque sa période et sa mesure correspondent à la question, développe les sigles usuels
(PIB, IPPI, ICA, FBCF), distingue niveau absolu, taux, part et sous-population, puis vérifie que
le sujet précis demandé est couvert (par exemple secteur primaire plutôt qu'un taux générique).
Une question d'effectif ne favorise pas un ratio et la colonne de fin est privilégiée pour une année
scolaire. Les cellules structurellement corrompues sont exclues de Dense, BM25 et du canal structuré
avant le MMR ; le passage textuel original reste interrogeable et le cross-encoder conserve son rôle.
Pour une part ou proportion explicite, deux cellules ne sont appariées que si période et échelle concordent. Elles restent dans le même tableau sauf si la question demande explicitement de rapporter une grandeur à une autre ; un rapprochement inter-document exige alors aussi le même territoire et conserve les deux sources. Le quotient est calculé par le code et non improvisé par le modèle.
Sur une valeur directe, le rang du canal structuré protège légèrement la meilleure cellule après le
reranking. Pour une comparaison, ce signal est désactivé et le passage qui exprime explicitement la
relation demandée (contre, situation de référence, écart) reçoit la priorité.

In [6]:
if RUN_RETRIEVAL:
    structured_checks = [
        {
            'question': "Combien d’hommes comptait la Côte d’Ivoire en 2025 ?",
            'indicator': 'population homme',
            'period': '2025',
            'value': '17 103 895',
        },
        {
            'question': 'Quel était le rapport de masculinité en 2025 ?',
            'indicator': 'rapport de masculinité',
            'period': '2025',
            'value': '109,3',
        },
    ]
    validation_rows = []
    for check in structured_checks:
        retrieved = pipeline.retriever.retrieve(check['question'])
        for rank, chunk in enumerate(retrieved, start=1):
            validation_rows.append({
                'question': check['question'],
                'rang': rank,
                'type': chunk.content_type,
                'page': chunk.page,
                'indicateur': chunk.indicator,
                'période': chunk.period,
                'valeur': chunk.value,
                'rerank': chunk.rerank_score,
                'attendu': (
                    chunk.content_type == 'table_fact'
                    and chunk.indicator.casefold() == check['indicator']
                    and chunk.period == check['period']
                    and chunk.value == check['value']
                ),
            })
    validation_df = pd.DataFrame(validation_rows)
    display(validation_df)
    expected_rows = validation_df[validation_df['attendu']]
    best_expected_rank = expected_rows.groupby('question')['rang'].min()
    missing = sorted(set(validation_df['question']) - set(best_expected_rank.index))
    assert not missing, f'Faits exacts absents du top final : {missing}'
    assert best_expected_rank.eq(1).all(), (
        'Le bon fait est retrouvé mais pas au rang 1 : ' + str(best_expected_rank.to_dict())
    )
    debug = pd.DataFrame(pipeline.retriever.last_debug_rows)
    useful = [column for column in (
        'title', 'content_type', 'page', 'indicator', 'period', 'value',
        'dense_rank', 'bm25_rank', 'table_fact_rank', 'page_rescue_rank', 'page_rescue_bonus', 'rrf_rank', 'facet_bonus', 'rerank_rank',
        'rerank_score', 'selection_score', 'final_rank',
    ) if column in debug]
    display(debug[useful].sort_values(['final_rank', 'rerank_rank'], na_position='last').head(30))

    calculation_checks = [
        {
            'question': 'Quel était le PIB en volume CVS au quatrième trimestre 2025 ?',
            'value': '12 083,3', 'type': 'table_fact', 'page': 14,
        },
        {
            'question': 'De combien le PIB en volume CVS a-t-il varié entre le troisième et le quatrième trimestre 2025 ?',
            'value': '-0,4', 'type': 'derived_fact', 'page': 14,
        },
        {
            'question': 'Quelle était la valeur nominale totale de l’agriculture vivrière sur l’ensemble de l’année 2025 ?',
            'value': '3357,6', 'type': 'derived_fact', 'page': 17,
        },
        {
            'question': 'En rapportant le déficit budgétaire 2025 au PIB nominal annuel 2025, quelle part du PIB représentait-il ?',
            'value': '2,9', 'type': 'derived_fact', 'page': 12,
        },
    ]
    calculation_rows = []
    for check in calculation_checks:
        retrieved = pipeline.retriever.retrieve(check['question'])
        matches = [
            (rank, chunk) for rank, chunk in enumerate(retrieved, start=1)
            if chunk.content_type == check['type']
            and chunk.value == check['value']
            and chunk.page == check['page']
        ]
        calculation_rows.append({
            'question': check['question'],
            'trouvé': bool(matches),
            'meilleur_rang': matches[0][0] if matches else None,
            'valeur': matches[0][1].value if matches else None,
            'page': matches[0][1].page if matches else None,
            'type': matches[0][1].content_type if matches else None,
            'page_rescue_candidats': pipeline.retriever.last_timings.get('page_rescue_candidates', 0),
        })
    calculation_df = pd.DataFrame(calculation_rows)
    display(calculation_df)
    assert calculation_df['trouvé'].all(), 'Au moins un fait direct ou calculé manque dans le top final.'

,question,rang,type,page,indicateur,période,valeur,rerank,attendu
0,Combien d’hommes comptait la Côte d’Ivoire en ...,1,table_fact,10,Population homme,2025,17 103 895,0.995768,True
1,Combien d’hommes comptait la Côte d’Ivoire en ...,2,table_fact,10,Espérance de vie à la naissance (hommes),2025,"60,4",0.742795,False
2,Combien d’hommes comptait la Côte d’Ivoire en ...,3,table_fact,297,Homme,2018,97%,0.936634,False
3,Combien d’hommes comptait la Côte d’Ivoire en ...,4,text,88,,,,0.925634,False
4,Combien d’hommes comptait la Côte d’Ivoire en ...,5,text,23,,,,0.837221,False
5,Quel était le rapport de masculinité en 2025 ?,1,table_fact,10,Rapport de masculinité,2025,"109,3",0.991187,True
6,Quel était le rapport de masculinité en 2025 ?,2,text,61,,,,0.516413,False
7,Quel était le rapport de masculinité en 2025 ?,3,text,115,,,,0.413655,False


,title,content_type,page,indicator,period,value,dense_rank,bm25_rank,table_fact_rank,page_rescue_rank,page_rescue_bonus,rrf_rank,facet_bonus,rerank_rank,rerank_score,selection_score,final_rank
0,Chiffres Cles CIV 2025,table_fact,10,Rapport de masculinité,2025,"109,3",1.0,1.0,1.0,None,0.0,1,1.035,1.0,0.991187,2.206187,1.0
20,RGPH 98TOME1,text,61,,,,NaN,15.0,NaN,None,0.0,21,0.000,3.0,0.516413,0.516413,2.0
14,RGPH 2014,text,115,,,,12.0,17.0,NaN,None,0.0,15,0.000,4.0,0.413655,0.413655,3.0
11,RGPH 2014,table_fact,7,Rapport de masculinité,1988,105,23.0,12.0,NaN,None,0.0,12,0.565,2.0,0.039788,0.604788,NaN
22,RGPH 98TOME1,text,47,,,,NaN,16.0,NaN,None,0.0,23,0.000,5.0,0.265309,0.265309,NaN
16,la cote d ivoire en chiffres edition 2025 92,text,266,,,,10.0,NaN,NaN,None,0.0,17,0.000,6.0,0.138812,0.138812,NaN
15,la cote d ivoire en chiffres edition 2025 92,text,296,,,,8.0,NaN,NaN,None,0.0,16,0.000,7.0,0.082994,0.082994,NaN
1,Chiffres Cles CIV 2025,table_fact,10,Rapport de masculinité,2022,"109,3",2.0,9.0,NaN,None,0.0,2,0.865,NaN,NaN,NaN,NaN
2,Chiffres Cles CIV 2025,table_fact,10,Rapport de masculinité,2015,"107,9",9.0,2.0,NaN,None,0.0,3,0.865,NaN,NaN,NaN,NaN
3,Chiffres Cles CIV 2025,table_fact,10,Rapport de masculinité,2023,"109,3",3.0,10.0,NaN,None,0.0,4,0.865,NaN,NaN,NaN,NaN


,question,trouvé,meilleur_rang,valeur,page,type,page_rescue_candidats
0,Quel était le PIB en volume CVS au quatrième t...,True,1.0,"12 083,3",14.0,table_fact,0
1,De combien le PIB en volume CVS a-t-il varié e...,False,NaN,NaN,NaN,NaN,0
2,Quelle était la valeur nominale totale de l’ag...,False,NaN,NaN,NaN,NaN,0
3,En rapportant le déficit budgétaire 2025 au PI...,False,NaN,NaN,NaN,NaN,0


AssertionError: Au moins un fait direct ou calculé manque dans le top final.